# Grounded-Physics LM Adapter Training

**Copyright (c) 2026 Style Machine LLC. All rights reserved.**

**Author:** Jesse Pokora

**PROPRIETARY AND CONFIDENTIAL.** This software is provided for academic review and research purposes only. Unauthorized copying, modification, distribution, or use of this software, via any medium, is strictly prohibited without prior written permission from Style Machine LLC.

---

This notebook trains the Grounded-Physics LM Adapter on top of a pre-trained PhysicsFormer model.

## Architecture
- **PhysicsFormer** (frozen): Encodes physics states into embeddings
- **Adapter MLP**: Projects physics embeddings to LLM prefix tokens
- **DistilGPT-2** (82M): Generates answers from prefix + question
- **Specialized Heads**: Numerical (6 outputs) + Descriptive (multi-head classification)

## Training Phases (Progressive Unfreezing)
1. **Phase 1**: Train adapter + heads (LLM frozen)
2. **Phase 2**: Unfreeze LLM output layer
3. **Phase 3**: Unfreeze full LLM

## Key Features (Physics Former V2 Technologies)
- **AMP (Automatic Mixed Precision)** - 2x faster on A100
- **OneCycleLR Scheduler** - Better convergence
- **PlateauTracker + Catapult** - Escape training plateaus
- **Contrastive Loss** - Prevent modality collapse (model ignoring physics)
- **Physics Usage Validation** - Detect if model actually uses physics
- **Plateau epochs don't count against progression** - Fair chance for catapult

## Prerequisites
- Trained PhysicsFormer checkpoint (from `train_physics_former_v2.ipynb`)
- Physics HDF5 data files

## Hardware Optimization
- **A100 80GB**: Batch size 64, 4 workers, pin memory enabled

In [10]:
# ============================================================
# CELL 1: CONFIGURATION (A100 80GB OPTIMIZED)
# ============================================================
# All data is in the pre-generated QA cache - no separate HDF5 files needed.

# GDrive base path for adapter data
GDRIVE_ADAPTER_DATA = "/content/drive/MyDrive/physics_action_predictor"

# Path to PhysicsFormer checkpoint
PHYSICS_CHECKPOINT = f"{GDRIVE_ADAPTER_DATA}/checkpoints/physics_former/physics_former_latest.pt"

# Pre-generated QA dataset (contains physics states + questions + answers)
QA_DATASET_PATH = f"{GDRIVE_ADAPTER_DATA}/data/physics_former_adapter/adapter_qa_cache.pt"

# Output directory for adapter checkpoints
OUTPUT_DIR = f"{GDRIVE_ADAPTER_DATA}/checkpoints/acheckpoints"

# CLEVRER data path (optional - for additional training data)
CLEVRER_DATA_PATH = f"{GDRIVE_ADAPTER_DATA}/data/physics_former_adapter/clevrer_conforming_qa.json"

# ============================================================
# A100 80GB OPTIMIZATIONS
# ============================================================
BATCH_SIZE = 64
GRADIENT_ACCUMULATION = 1
NUM_WORKERS = 4
PIN_MEMORY = True

NUM_PREFIX_TOKENS = 64

# Physics model dimensions (must match train_physics_former_v2.ipynb)
STATE_DIM = 28
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 6
FF_DIM = 1024
MAX_OBJECTS = 20

# Phase-specific learning rates
LR_SCALE = (BATCH_SIZE / 8) ** 0.5
PHASE1_LR = 1e-4 * LR_SCALE
PHASE2_LR = 5e-5 * LR_SCALE
PHASE3_LR = 2e-5 * LR_SCALE

# Contrastive loss
USE_CONTRASTIVE = True
CONTRASTIVE_WEIGHT = 0.1

print("="*70)
print("A100 80GB OPTIMIZED CONFIGURATION")
print("="*70)
print(f"GDrive data path: {GDRIVE_ADAPTER_DATA}")
print(f"Physics checkpoint: {PHYSICS_CHECKPOINT}")
print(f"QA dataset (contains states): {QA_DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print()
print(f"Batch size: {BATCH_SIZE}, LR scale: {LR_SCALE:.2f}x")
print(f"Physics: state_dim={STATE_DIM}, embed_dim={EMBED_DIM}")
print("="*70)

A100 80GB OPTIMIZED CONFIGURATION
GDrive data path: /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former
Physics checkpoint: /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former/checkpoints/physics_former_latest.pt
QA dataset (contains states): /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former/adapter_qa_cache.pt
Output directory: /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former/adapter_checkpoints

Batch size: 64, LR scale: 2.83x
Physics: state_dim=28, embed_dim=256


In [11]:
# ============================================================
# CELL 2: IMPORTS AND SETUP
# ============================================================
# Fully self-contained - no companion bundle needed.

import os
import sys
import json
import subprocess
import math
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
import h5py

# Mount Google Drive if on Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
    print("Running on Google Colab")
except:
    ON_COLAB = False
    print("Running locally")

# Install required external deps
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "tqdm", "h5py"
])

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Google Colab
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB


In [12]:
# ============================================================
# CELL 3: PHYSICS CONFIG
# ============================================================
# This must match the config used to train the PhysicsFormer checkpoint

@dataclass
class PhysicsConfig:
    """Configuration for PhysicsFormer V2."""
    # Model architecture
    num_objects: int = 20
    state_dim: int = 28  # Reduced: skipped quaternion (4D) + angular velocity (3D)
    embed_dim: int = 256
    num_heads: int = 8
    num_layers: int = 6
    mlp_ratio: float = 4.0
    ff_dim: int = 1024
    dropout: float = 0.1
    max_seq_len: int = 128

    # Modern improvements
    use_rope: bool = True
    use_rmsnorm: bool = True
    use_swiglu: bool = True
    use_flash_attention: bool = True

    # Physics-specific
    use_physics_bias: bool = False
    use_learned_scaling: bool = True
    use_graph_attention: bool = False
    use_masked_attention: bool = True
    graph_edge_type: str = "spatial"
    spatial_threshold: float = 2.0
    use_hadamard_attention: bool = False
    per_feature_attention: bool = True
    use_energy_conservation: bool = False
    hamiltonian_weight: float = 0.05

physics_config = PhysicsConfig()
print(f"PhysicsConfig: state_dim={physics_config.state_dim}, embed_dim={physics_config.embed_dim}")

PhysicsConfig: state_dim=28, embed_dim=256


In [13]:
# ============================================================
# CELL 4: PHYSICSFORMER V2 MODEL
# ============================================================
# Minimal model for adapter inference - no training heads.
# Only includes encode_physics() method needed by the adapter.

# -----------------------------------------------------------------------------
# RoPE (Rotary Position Embedding)
# -----------------------------------------------------------------------------
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 512, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, device=self.inv_freq.device)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cache', emb.cos().unsqueeze(0).unsqueeze(0))
        self.register_buffer('sin_cache', emb.sin().unsqueeze(0).unsqueeze(0))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = x.shape[2]
        if seq_len > self.max_seq_len:
            self._build_cache(seq_len)
        return self.cos_cache[:, :, :seq_len], self.sin_cache[:, :, :seq_len]

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

# -----------------------------------------------------------------------------
# RMSNorm
# -----------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x / torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps) * self.weight

# -----------------------------------------------------------------------------
# SwiGLU Activation
# -----------------------------------------------------------------------------
class SwiGLU(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int, bias: bool = False):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.w2 = nn.Linear(hidden_features, out_features, bias=bias)
        self.w3 = nn.Linear(in_features, hidden_features, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

# -----------------------------------------------------------------------------
# State Encoder
# -----------------------------------------------------------------------------
class StateEncoder(nn.Module):
    def __init__(self, config: PhysicsConfig):
        super().__init__()
        self.input_scale = nn.Parameter(torch.ones(config.state_dim))
        self.input_bias = nn.Parameter(torch.zeros(config.state_dim))
        self.encoder = nn.Sequential(
            nn.Linear(config.state_dim, config.embed_dim),
            RMSNorm(config.embed_dim),
            nn.GELU(),
            nn.Dropout(config.dropout)
        )

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        states = torch.clamp(states, min=-100.0, max=100.0)
        scale = torch.clamp(self.input_scale, min=0.01, max=10.0)
        bias = torch.clamp(self.input_bias, min=-10.0, max=10.0)
        states = torch.clamp(states * scale + bias, min=-100.0, max=100.0)
        return self.encoder(states)

# -----------------------------------------------------------------------------
# Physics Attention V2
# -----------------------------------------------------------------------------
class PhysicsAttentionV2(nn.Module):
    def __init__(self, config: PhysicsConfig):
        super().__init__()
        self.num_heads = config.num_heads
        self.head_dim = config.embed_dim // config.num_heads
        self.q_proj = nn.Linear(config.embed_dim, config.embed_dim)
        self.k_proj = nn.Linear(config.embed_dim, config.embed_dim)
        self.v_proj = nn.Linear(config.embed_dim, config.embed_dim)
        self.o_proj = nn.Linear(config.embed_dim, config.embed_dim)
        self.rope = RotaryPositionEmbedding(self.head_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, physics_states=None, mask=None):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(q)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        attn_mask = None
        if mask is not None and mask.dim() == 2:
            key_mask = mask.unsqueeze(1).unsqueeze(2)
            attn_mask = torch.zeros(B, 1, 1, N, device=q.device, dtype=q.dtype)
            attn_mask = attn_mask.masked_fill(~key_mask.bool(), float('-inf'))
            attn_mask = attn_mask.expand(-1, self.num_heads, N, -1)

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
            dropout_p=self.dropout.p if self.training else 0.0, is_causal=False)
        return self.o_proj(out.transpose(1, 2).contiguous().view(B, N, C))

# -----------------------------------------------------------------------------
# Physics Transformer Block V2
# -----------------------------------------------------------------------------
class PhysicsTransformerBlockV2(nn.Module):
    def __init__(self, config: PhysicsConfig, layer_idx: int = 0):
        super().__init__()
        self.residual_scale = 1.0 / math.sqrt(config.num_layers)
        self.norm1 = RMSNorm(config.embed_dim)
        self.norm2 = RMSNorm(config.embed_dim)
        self.attn = PhysicsAttentionV2(config)
        swiglu_hidden = int(config.ff_dim * 2 / 3)
        self.mlp = SwiGLU(config.embed_dim, swiglu_hidden, config.embed_dim, bias=True)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, physics_states=None, mask=None):
        x = x + self.residual_scale * self.dropout(self.attn(self.norm1(x), physics_states, mask))
        x = x + self.residual_scale * self.dropout(self.mlp(self.norm2(x)))
        return x

# -----------------------------------------------------------------------------
# PhysicsFormer V2 (Adapter-Only - No Training Heads)
# -----------------------------------------------------------------------------
class PhysicsFormerV2(nn.Module):
    """PhysicsFormer V2 for adapter inference only. No training heads."""

    def __init__(self, config: PhysicsConfig):
        super().__init__()
        self.config = config
        self.state_encoder = StateEncoder(config)
        self.blocks = nn.ModuleList([
            PhysicsTransformerBlockV2(config, i) for i in range(config.num_layers)
        ])
        self.norm = RMSNorm(config.embed_dim)
        self.output_proj = nn.Linear(config.embed_dim, config.embed_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode_physics(self, x, mask=None):
        """Get per-object physics embeddings (used by adapter)."""
        hidden = self.state_encoder(x)
        for block in self.blocks:
            hidden = block(hidden, x, mask)
        return self.norm(hidden)  # [batch, num_objects, embed_dim]

    def encode(self, x, mask=None):
        """Get global physics embedding (pooled)."""
        hidden = self.encode_physics(x, mask)
        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).float()
            return self.output_proj((hidden * mask_expanded).sum(dim=1) / (mask_expanded.sum(dim=1) + 1e-8))
        return self.output_proj(hidden.mean(dim=1))

print("PhysicsFormerV2 (adapter-only) defined - no training heads")

PhysicsFormerV2 (adapter-only) defined - no training heads


In [14]:
# ============================================================
# CELL 5: ADAPTER ENUMS AND VOCABULARIES
# ============================================================

class OutputType(Enum):
    CATEGORICAL = "categorical"
    NUMERICAL = "numerical"
    DESCRIPTIVE = "descriptive"

# CLEVRER descriptive answer vocabulary (21 classes)
CLEVRER_DESCRIPTIVE_VOCAB = [
    "0", "1", "2", "3", "4", "5",  # Numbers (count)
    "yes", "no",  # Yes/No (exist)
    "brown", "red", "purple", "cyan", "gray", "green", "blue", "yellow",  # Colors
    "cylinder", "sphere", "cube",  # Shapes
    "metal", "rubber"  # Materials
]
CLEVRER_ANSWER_TO_IDX = {ans: idx for idx, ans in enumerate(CLEVRER_DESCRIPTIVE_VOCAB)}
CLEVRER_IDX_TO_ANSWER = {idx: ans for idx, ans in enumerate(CLEVRER_DESCRIPTIVE_VOCAB)}

class DescriptiveSubtype(Enum):
    COUNT = "count"
    EXIST = "exist"
    QUERY_COLOR = "color"
    QUERY_SHAPE = "shape"
    QUERY_MATERIAL = "material"

SUBTYPE_VOCABS = {
    DescriptiveSubtype.COUNT: ["0", "1", "2", "3", "4", "5"],
    DescriptiveSubtype.EXIST: ["yes", "no"],
    DescriptiveSubtype.QUERY_COLOR: ["brown", "red", "purple", "cyan", "gray", "green", "blue", "yellow"],
    DescriptiveSubtype.QUERY_SHAPE: ["cylinder", "sphere", "cube"],
    DescriptiveSubtype.QUERY_MATERIAL: ["metal", "rubber"],
}

def classify_descriptive_subtype(question_text: str) -> DescriptiveSubtype:
    q_lower = question_text.lower().strip()
    if "how many" in q_lower:
        return DescriptiveSubtype.COUNT
    elif "what color" in q_lower or "what is the color" in q_lower:
        return DescriptiveSubtype.QUERY_COLOR
    elif "what shape" in q_lower or "what is the shape" in q_lower:
        return DescriptiveSubtype.QUERY_SHAPE
    elif "what material" in q_lower or "what is the material" in q_lower:
        return DescriptiveSubtype.QUERY_MATERIAL
    elif any(p in q_lower for p in ["is there", "are there", "any"]):
        return DescriptiveSubtype.EXIST
    return DescriptiveSubtype.EXIST

class CLEVRERQuestionCategory(Enum):
    DESCRIPTIVE = "descriptive"
    EXPLANATORY = "explanatory"
    PREDICTIVE = "predictive"
    COUNTERFACTUAL = "counterfactual"

DESCRIPTIVE_PATTERNS = ["how many", "what color", "what shape", "what material",
    "what is the color", "what is the shape", "what is the material",
    "are there any", "is there a", "are there", "is there"]
EXPLANATORY_PATTERNS = ["what caused", "responsible for", "why did", "what made",
    "which of the following is responsible"]
PREDICTIVE_PATTERNS = ["what will happen", "which event will happen", "will the", "what happens next"]
COUNTERFACTUAL_PATTERNS = ["what if", "without the", "if the .* is removed",
    "if the .* were removed", "if we remove"]

def classify_clevrer_question(question_text: str) -> CLEVRERQuestionCategory:
    q_lower = question_text.lower().strip()
    for pattern in COUNTERFACTUAL_PATTERNS:
        if re.search(pattern, q_lower):
            return CLEVRERQuestionCategory.COUNTERFACTUAL
    for pattern in EXPLANATORY_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.EXPLANATORY
    for pattern in PREDICTIVE_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.PREDICTIVE
    for pattern in DESCRIPTIVE_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.DESCRIPTIVE
    return CLEVRERQuestionCategory.DESCRIPTIVE

print("Adapter enums and vocabularies defined")

Adapter enums and vocabularies defined


In [15]:
# ============================================================
# CELL 6: ADAPTER HEADS (Descriptive + Numerical)
# ============================================================

class DescriptiveSubHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.num_classes = num_classes
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

class DescriptiveHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.count_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=6)
        self.exist_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=2)
        self.color_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=8)
        self.shape_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=3)
        self.material_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=2)
        self.subtype_vocabs = SUBTYPE_VOCABS
        for m in self.shared_encoder.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _get_head_for_subtype(self, subtype):
        return {DescriptiveSubtype.COUNT: self.count_head, DescriptiveSubtype.EXIST: self.exist_head,
                DescriptiveSubtype.QUERY_COLOR: self.color_head, DescriptiveSubtype.QUERY_SHAPE: self.shape_head,
                DescriptiveSubtype.QUERY_MATERIAL: self.material_head}.get(subtype, self.exist_head)

    def forward(self, physics_features, subtype=None):
        shared = self.shared_encoder(physics_features)
        if subtype is None:
            return shared
        return self._get_head_for_subtype(subtype)(shared)

    def predict_batch(self, physics_features, question_texts):
        subtypes = [classify_descriptive_subtype(q) for q in question_texts]
        if len(set(subtypes)) == 1:
            subtype = subtypes[0]
            logits = self.forward(physics_features, subtype)
            pred_indices = torch.argmax(logits, dim=-1).tolist()
            vocab = self.subtype_vocabs[subtype]
            return [vocab[idx] for idx in pred_indices]
        answers = []
        for i, (subtype, q) in enumerate(zip(subtypes, question_texts)):
            logits = self.forward(physics_features[i:i+1], subtype)
            pred_idx = torch.argmax(logits, dim=-1).item()
            answers.append(self.subtype_vocabs[subtype][pred_idx])
        return answers

    def compute_loss(self, physics_features, question_texts, answer_texts, label_smoothing=0.1):
        total_loss = torch.tensor(0.0, device=physics_features.device)
        count = 0
        for i, (q, a) in enumerate(zip(question_texts, answer_texts)):
            subtype = classify_descriptive_subtype(q)
            logits = self.forward(physics_features[i:i+1], subtype)
            vocab = self.subtype_vocabs[subtype]
            a_lower = a.lower().strip()
            if a_lower in vocab:
                target = torch.tensor([vocab.index(a_lower)], device=logits.device)
                total_loss = total_loss + F.cross_entropy(logits, target, label_smoothing=label_smoothing)
                count += 1
        return total_loss / max(count, 1)

class NumericalHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, num_outputs: int = 6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_outputs)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, physics_features):
        return self.network(physics_features)

print("DescriptiveHead and NumericalHead defined")

DescriptiveHead and NumericalHead defined


In [16]:
# ============================================================
# CELL 7: GROUNDED PHYSICS LM (Main Class)
# ============================================================

class GroundedPhysicsLM(nn.Module):
    """
    Grounded Physics Language Model

    Combines physics understanding with language generation:
    - PhysicsFormer encodes physics states into embeddings
    - Adapter projects physics embeddings to LLM prefix tokens
    - DistilGPT-2 generates answers from prefix + question
    - Specialized heads for numerical and descriptive outputs
    """

    LLM_CONFIGS = {
        "distilgpt2": {"dim": 768, "params": "82M"},
        "gpt2": {"dim": 768, "params": "124M"},
    }

    def __init__(self, physics_model, physics_dim: int = 768, llm_name: str = "distilgpt2",
                 num_prefix_tokens: int = 64, freeze_physics: bool = True, freeze_llm: bool = True,
                 input_noise: float = 0.01):
        super().__init__()
        if llm_name not in self.LLM_CONFIGS:
            raise ValueError(f"Unknown LLM: {llm_name}")

        self.llm_name = llm_name
        self.llm_dim = self.LLM_CONFIGS[llm_name]["dim"]
        self.physics_dim = physics_dim
        self.num_prefix_tokens = num_prefix_tokens
        self.input_noise = input_noise

        self.physics_model = physics_model
        if freeze_physics:
            self.physics_model.eval()
            for param in self.physics_model.parameters():
                param.requires_grad = False

        self.adapter = nn.Sequential(
            nn.Linear(physics_dim, self.llm_dim),
            nn.LayerNorm(self.llm_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.llm_dim, self.llm_dim * num_prefix_tokens),
            nn.LayerNorm(self.llm_dim * num_prefix_tokens),
            nn.Tanh()
        )
        for m in self.adapter.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.numerical_head = NumericalHead(input_dim=physics_dim, hidden_dim=768, num_outputs=6)
        self.descriptive_head = DescriptiveHead(input_dim=physics_dim, hidden_dim=512)

        print(f"Loading LLM: {self.llm_name}...")
        self.llm = AutoModelForCausalLM.from_pretrained(self.llm_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.llm_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        if freeze_llm:
            self.llm.eval()
            for param in self.llm.parameters():
                param.requires_grad = False

        print(f"GroundedPhysicsLM: LLM={llm_name}, physics_dim={physics_dim}, prefix_tokens={num_prefix_tokens}")

    def extract_physics_features(self, physics_states, object_mask):
        if self.training and self.input_noise > 0:
            physics_states = physics_states + torch.randn_like(physics_states) * self.input_noise

        with torch.no_grad() if not self.physics_model.training else torch.enable_grad():
            # Pass physics states directly (no augmentation needed)

            obj_emb = self.physics_model.encode_physics(physics_states, object_mask)

            if obj_emb.dim() == 4:
                if object_mask is not None:
                    mask_exp = object_mask.unsqueeze(1).unsqueeze(-1) if object_mask.dim() == 2 else object_mask.unsqueeze(-1)
                    obj_emb = obj_emb * mask_exp
                    features = obj_emb.sum(dim=2) / object_mask.sum(dim=-1, keepdim=True).clamp(min=1).unsqueeze(-1)
                    features = features.mean(dim=1)
                else:
                    features = obj_emb.mean(dim=(1, 2))
            elif obj_emb.dim() == 3:
                if object_mask is not None:
                    mask_exp = object_mask.unsqueeze(-1) if object_mask.dim() == 2 else object_mask[:, 0, :].unsqueeze(-1)
                    features = (obj_emb * mask_exp).sum(dim=1) / object_mask.sum(dim=-1, keepdim=True).clamp(min=1)
                else:
                    features = obj_emb.mean(dim=1)
            else:
                features = obj_emb
            return features

    def create_prefix_tokens(self, physics_features):
        batch_size = physics_features.size(0)
        return self.adapter(physics_features).view(batch_size, self.num_prefix_tokens, self.llm_dim)

    def predict_numerical(self, physics_states, object_mask):
        features = self.extract_physics_features(physics_states, object_mask)
        outputs = self.numerical_head(features)
        return {"distance": outputs[:, 0], "speed": outputs[:, 1], "time_to_collision": outputs[:, 2],
                "kinetic_energy": outputs[:, 3], "momentum": outputs[:, 4], "object_count": outputs[:, 5]}

    def _compute_question_lengths(self, question_text):
        return [len(self.tokenizer.encode(q + " Answer:", add_special_tokens=False)) for q in question_text]

    def forward(self, physics_states, object_mask, question_text, max_length=50):
        batch_size = physics_states.size(0)
        device = physics_states.device

        features = self.extract_physics_features(physics_states, object_mask)
        prefix = self.create_prefix_tokens(features)

        prompted = [q + " Answer:" for q in question_text]
        tokens = self.tokenizer(prompted, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        q_embeds = self.llm.transformer.wte(tokens.input_ids)
        combined = torch.cat([prefix, q_embeds], dim=1)
        mask = torch.cat([torch.ones(batch_size, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

        outputs = self.llm.generate(inputs_embeds=combined, attention_mask=mask, max_new_tokens=max_length,
                                    temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)

        answers = []
        for text in self.tokenizer.batch_decode(outputs, skip_special_tokens=True):
            answers.append(text.split("Answer:")[-1].strip() if "Answer:" in text else text.strip())
        return answers

print("GroundedPhysicsLM defined")

GroundedPhysicsLM defined


In [17]:
# ============================================================
# CELL 8: GROUNDED PHYSICS LM - LOSS AND TRAINING METHODS
# ============================================================

# Add methods to GroundedPhysicsLM
def _adapter_compute_loss(self, physics_states, object_mask, question_text, answer_text):
    batch_size = physics_states.size(0)
    device = physics_states.device
    features = self.extract_physics_features(physics_states, object_mask)
    prefix = self.create_prefix_tokens(features)

    full_text = [q + " Answer: " + a for q, a in zip(question_text, answer_text)]
    tokens = self.tokenizer(full_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    q_lengths = self._compute_question_lengths(question_text)

    text_embeds = self.llm.transformer.wte(tokens.input_ids)
    combined = torch.cat([prefix, text_embeds], dim=1)
    mask = torch.cat([torch.ones(batch_size, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

    labels = tokens.input_ids.clone()
    for i in range(batch_size):
        labels[i, :q_lengths[i]] = -100
    labels = torch.cat([torch.full((batch_size, self.num_prefix_tokens), -100, dtype=torch.long, device=device), labels], dim=1)

    return self.llm(inputs_embeds=combined, attention_mask=mask, labels=labels).loss

def _adapter_compute_numerical_loss(self, physics_states, object_mask, targets):
    preds = self.predict_numerical(physics_states, object_mask)
    total_loss, count = 0.0, 0
    for key in targets:
        if key in preds:
            total_loss += F.mse_loss(preds[key], targets[key].to(preds[key].device))
            count += 1
    return total_loss / max(count, 1)

def _adapter_answer_descriptive(self, physics_states, object_mask, question_text):
    features = self.extract_physics_features(physics_states, object_mask)
    return self.descriptive_head.predict_batch(features, question_text)

def _adapter_compute_descriptive_loss(self, physics_states, object_mask, question_text, answer_text, label_smoothing=0.1):
    features = self.extract_physics_features(physics_states, object_mask)
    return self.descriptive_head.compute_loss(features, question_text, answer_text, label_smoothing)

def _adapter_score_answer_candidates(self, physics_states, object_mask, question_text, answer_candidates, max_length=128):
    batch_size = physics_states.size(0)
    device = physics_states.device
    k = len(answer_candidates[0])

    features = self.extract_physics_features(physics_states, object_mask)
    prefix = self.create_prefix_tokens(features)
    expanded_prefix = prefix.unsqueeze(1).expand(-1, k, -1, -1).reshape(batch_size * k, self.num_prefix_tokens, self.llm_dim)

    expanded_questions, expanded_full = [], []
    for q, choices in zip(question_text, answer_candidates):
        for a in choices:
            expanded_questions.append(q)
            expanded_full.append(q + " Answer: " + a)

    tokens = self.tokenizer(expanded_full, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    q_lengths = self._compute_question_lengths(expanded_questions)

    text_embeds = self.llm.transformer.wte(tokens.input_ids)
    combined = torch.cat([expanded_prefix, text_embeds], dim=1)
    mask = torch.cat([torch.ones(batch_size * k, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

    labels = tokens.input_ids.clone()
    for i in range(batch_size * k):
        labels[i, :q_lengths[i]] = -100
    labels = torch.cat([torch.full((batch_size * k, self.num_prefix_tokens), -100, dtype=torch.long, device=device), labels], dim=1)

    outputs = self.llm(inputs_embeds=combined, attention_mask=mask, labels=labels, return_dict=True)
    logits = outputs.logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    per_token_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), shift_labels.view(-1), reduction='none').view(batch_size * k, -1)
    valid_mask = (shift_labels != -100).float()
    per_sample_loss = (per_token_loss * valid_mask).sum(dim=-1) / (valid_mask.sum(dim=-1) + 1e-8)

    return (-per_sample_loss).reshape(batch_size, k)

def _adapter_select_answer_from_choices(self, physics_states, object_mask, question_text, choices, max_length=128):
    scores = self.score_answer_candidates(physics_states, object_mask, question_text, choices, max_length)
    best = torch.argmax(scores, dim=-1).tolist()
    return [c[i] for c, i in zip(choices, best)]

def _adapter_compute_contrastive_loss(self, physics_states, object_mask, temperature=0.07):
    batch_size = physics_states.size(0)
    if batch_size < 1:
        return torch.tensor(0.0, device=physics_states.device)

    real_features = self.extract_physics_features(physics_states, object_mask)
    zero_features = self.extract_physics_features(torch.zeros_like(physics_states), object_mask)

    real_prefix = self.create_prefix_tokens(real_features).view(batch_size, -1)
    zero_prefix = self.create_prefix_tokens(zero_features).view(batch_size, -1)

    real_norm = F.normalize(real_prefix, p=2, dim=1)
    zero_norm = F.normalize(zero_prefix, p=2, dim=1)
    cos_sim = (real_norm * zero_norm).sum(dim=1)

    return F.relu(cos_sim - 0.5).mean()

def _adapter_compute_combined_loss(self, physics_states, object_mask, questions, answers,
                                    choices=None, correct_choice_idx=None, numerical_targets=None,
                                    categorical_weight=1.0, numerical_weight=0.5):
    """Compute combined loss for training."""
    device = physics_states.device
    loss_dict = {}
    total_loss = torch.tensor(0.0, device=device)

    # Categorical loss (LLM generation)
    cat_loss = self.compute_loss(physics_states, object_mask, questions, answers)
    loss_dict['categorical'] = cat_loss
    total_loss = total_loss + categorical_weight * cat_loss

    # Numerical loss
    if numerical_targets:
        num_loss = self.compute_numerical_loss(physics_states, object_mask, numerical_targets)
        loss_dict['numerical'] = num_loss
        total_loss = total_loss + numerical_weight * num_loss
    else:
        loss_dict['numerical'] = torch.tensor(0.0, device=device)

    # Descriptive loss
    desc_loss = self.compute_descriptive_loss(physics_states, object_mask, questions, answers)
    loss_dict['descriptive'] = desc_loss
    total_loss = total_loss + 0.3 * desc_loss

    return total_loss, loss_dict

def _adapter_compute_combined_loss_with_contrastive(self, physics_states, object_mask, questions, answers,
                                                     choices=None, correct_choice_idx=None, numerical_targets=None,
                                                     categorical_weight=1.0, numerical_weight=0.5, contrastive_weight=0.1):
    """Compute combined loss with contrastive term to prevent modality collapse."""
    total_loss, loss_dict = self.compute_combined_loss(
        physics_states, object_mask, questions, answers,
        choices=choices, correct_choice_idx=correct_choice_idx,
        numerical_targets=numerical_targets,
        categorical_weight=categorical_weight, numerical_weight=numerical_weight
    )

    # Add contrastive loss
    contr_loss = self.compute_contrastive_loss(physics_states, object_mask)
    loss_dict['contrastive'] = contr_loss
    total_loss = total_loss + contrastive_weight * contr_loss

    return total_loss, loss_dict

def _adapter_set_training_phase(self, phase):
    for model in [self.physics_model, self.adapter, self.numerical_head, self.descriptive_head, self.llm]:
        model.eval()
        for param in model.parameters():
            param.requires_grad = False

    if phase == 'adapter':
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head]:
            for param in m.parameters():
                param.requires_grad = True
    elif phase == 'llm_head':
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        self.llm.lm_head.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head]:
            for param in m.parameters():
                param.requires_grad = True
        for param in self.llm.lm_head.parameters():
            param.requires_grad = True
    elif phase in ['llm_full', 'full']:
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        self.llm.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head, self.llm]:
            for param in m.parameters():
                param.requires_grad = True
    print(f"[PHASE] {phase}")

# Attach methods to class
GroundedPhysicsLM.compute_loss = _adapter_compute_loss
GroundedPhysicsLM.compute_numerical_loss = _adapter_compute_numerical_loss
GroundedPhysicsLM.answer_descriptive = _adapter_answer_descriptive
GroundedPhysicsLM.compute_descriptive_loss = _adapter_compute_descriptive_loss
GroundedPhysicsLM.score_answer_candidates = _adapter_score_answer_candidates
GroundedPhysicsLM.select_answer_from_choices = _adapter_select_answer_from_choices
GroundedPhysicsLM.compute_contrastive_loss = _adapter_compute_contrastive_loss
GroundedPhysicsLM.compute_combined_loss = _adapter_compute_combined_loss
GroundedPhysicsLM.compute_combined_loss_with_contrastive = _adapter_compute_combined_loss_with_contrastive
GroundedPhysicsLM.set_training_phase = _adapter_set_training_phase

def create_grounded_physics_lm(physics_model, **kwargs):
    """Factory function to create GroundedPhysicsLM."""
    return GroundedPhysicsLM(physics_model=physics_model, **kwargs)

print("GroundedPhysicsLM methods attached (including compute_combined_loss_with_contrastive)")

GroundedPhysicsLM methods attached (including compute_combined_loss_with_contrastive)


In [18]:
# ============================================================
# CELL 9: LOAD PHYSICS MODEL CHECKPOINT
# ============================================================
# PhysicsFormer checkpoint must be in GDrive. No fallbacks.

print("Loading PhysicsFormer V2 checkpoint...")

physics_checkpoint_path = Path(PHYSICS_CHECKPOINT)
if not physics_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Physics checkpoint not found at: {physics_checkpoint_path}\n\n"
        f"Upload physics_former_latest.pt to GDrive:\n"
        f"  {GDRIVE_ADAPTER_DATA}/checkpoints/physics_former_latest.pt"
    )

checkpoint = torch.load(physics_checkpoint_path, map_location=device, weights_only=False)

if 'model_state_dict' not in checkpoint:
    raise KeyError(f"Checkpoint missing 'model_state_dict' key. Keys: {list(checkpoint.keys())}")

model_state = checkpoint['model_state_dict']

# Handle _orig_mod prefix from torch.compile
has_orig_mod = any(k.startswith('_orig_mod.') for k in model_state.keys())
if has_orig_mod:
    model_state = {k.replace('_orig_mod.', ''): v for k, v in model_state.items()}
    print("  Removed _orig_mod. prefix from checkpoint keys")

# Filter out RoPE cached tensors (will be rebuilt)
model_state = {k: v for k, v in model_state.items()
               if 'cos_cache' not in k and 'sin_cache' not in k}

print(f"Checkpoint loaded: {len(model_state)} parameters")

Loading PhysicsFormer V2 checkpoint...


FileNotFoundError: Physics checkpoint not found at: /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former/checkpoints/physics_former_latest.pt

Upload physics_former_latest.pt to GDrive:
  /content/drive/MyDrive/physics_action_predictor/checkpoints/physics_former/checkpoints/physics_former_latest.pt

In [ ]:
# ============================================================
# CELL 10: CREATE PHYSICS MODEL
# ============================================================
# Create PhysicsFormerV2 using the inline definition from Cell 4
# This matches the exact architecture used in train_physics_former_v2.ipynb

physics_model = PhysicsFormerV2(physics_config).to(device)

# Load checkpoint weights
missing_keys, unexpected_keys = physics_model.load_state_dict(model_state, strict=False)

if missing_keys:
    print(f"Missing keys (expected for new heads): {len(missing_keys)}")
    for k in missing_keys[:5]:
        print(f"  - {k}")
    if len(missing_keys) > 5:
        print(f"  ... and {len(missing_keys) - 5} more")

if unexpected_keys:
    print(f"Unexpected keys (from checkpoint): {len(unexpected_keys)}")
    for k in unexpected_keys[:5]:
        print(f"  - {k}")
    if len(unexpected_keys) > 5:
        print(f"  ... and {len(unexpected_keys) - 5} more")

# Freeze physics model for adapter training
physics_model.eval()
for param in physics_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in physics_model.parameters())
print(f"\nPhysicsFormerV2 loaded: {total_params:,} parameters (frozen)")

In [ ]:
# ============================================================
# CELL 11: CREATE GROUNDED PHYSICS LM
# ============================================================
# This cell creates the model AFTER loading the physics checkpoint.

print("Creating GroundedPhysicsLM with DistilGPT-2...")

model = create_grounded_physics_lm(
    physics_model=physics_model,
    physics_dim=physics_config.embed_dim,  # From config (256)
    llm_name="distilgpt2",   # 82M params - edge-optimized
    num_prefix_tokens=NUM_PREFIX_TOKENS,
    freeze_physics=True,
    freeze_llm=True  # Will unfreeze progressively
).to(device)

# Alias for backward compatibility with training cells
adapter = model

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nGroundedPhysicsLM created:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Prefix tokens: {NUM_PREFIX_TOKENS}")
print(f"  Physics dim: {physics_config.embed_dim}")
print(f"  LLM: DistilGPT-2 (82M params)")

In [ ]:
# ============================================================
# CELL 12: LOAD PRE-GENERATED QA DATASET AND CREATE DATALOADERS
# ============================================================
# QA dataset must be pre-generated and stored in GDrive.
# Path: physics_action_predictor/data/physics_former_adapter/adapter_qa_cache.pt

import pickle
from torch.utils.data import Dataset, DataLoader, random_split

qa_cache_path = Path(QA_DATASET_PATH)

if not qa_cache_path.exists():
    raise FileNotFoundError(
        f"QA dataset not found at: {qa_cache_path}\n\n"
        f"Upload adapter_qa_cache.pt to GDrive:\n"
        f"  {GDRIVE_ADAPTER_DATA}/adapter_qa_cache.pt"
    )

print(f"Loading QA dataset from: {qa_cache_path}")

if qa_cache_path.suffix == '.pt':
    qa_dataset_raw = torch.load(qa_cache_path, map_location='cpu', weights_only=False)
elif qa_cache_path.suffix == '.pkl':
    with open(qa_cache_path, 'rb') as f:
        qa_dataset_raw = pickle.load(f)
elif qa_cache_path.suffix == '.json':
    with open(qa_cache_path, 'r') as f:
        qa_dataset_raw = json.load(f)
else:
    raise ValueError(f"Unsupported QA dataset format: {qa_cache_path.suffix}")

if isinstance(qa_dataset_raw, dict):
    if 'samples' in qa_dataset_raw:
        qa_dataset_raw = qa_dataset_raw['samples']
    elif 'data' in qa_dataset_raw:
        qa_dataset_raw = qa_dataset_raw['data']
    else:
        raise ValueError(f"QA dataset dict missing 'samples' or 'data' key. Keys: {list(qa_dataset_raw.keys())}")

print(f"QA Dataset loaded: {len(qa_dataset_raw):,} samples")

# ============================================================
# DATASET CLASS FOR ADAPTER TRAINING
# ============================================================
class AdapterQADataset(Dataset):
    """Dataset for adapter training from pre-generated QA cache."""
    
    def __init__(self, samples, max_objects=20, state_dim=28):
        self.samples = samples
        self.max_objects = max_objects
        self.state_dim = state_dim
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Get physics states
        if 'states' in sample:
            states = sample['states']
        elif 'physics_states' in sample:
            states = sample['physics_states']
        else:
            # Create dummy states if not present
            states = torch.zeros(self.max_objects, self.state_dim)
        
        # Convert to tensor if needed
        if not isinstance(states, torch.Tensor):
            states = torch.tensor(states, dtype=torch.float32)
        
        # Ensure correct shape [max_objects, state_dim]
        if states.dim() == 1:
            states = states.view(self.max_objects, -1)
        if states.dim() == 3:
            states = states[0]  # Take first timestep if sequence
        
        # Pad/truncate objects
        if states.shape[0] < self.max_objects:
            pad = torch.zeros(self.max_objects - states.shape[0], states.shape[1])
            states = torch.cat([states, pad], dim=0)
        elif states.shape[0] > self.max_objects:
            states = states[:self.max_objects]
        
        # Pad/truncate state_dim
        if states.shape[1] < self.state_dim:
            pad = torch.zeros(states.shape[0], self.state_dim - states.shape[1])
            states = torch.cat([states, pad], dim=1)
        elif states.shape[1] > self.state_dim:
            states = states[:, :self.state_dim]
        
        # Get object mask
        if 'mask' in sample:
            mask = sample['mask']
            if not isinstance(mask, torch.Tensor):
                mask = torch.tensor(mask, dtype=torch.float32)
        else:
            # Create mask from non-zero states
            mask = (states.abs().sum(dim=1) > 0).float()
        
        # Ensure mask is correct length
        if len(mask) < self.max_objects:
            mask = torch.cat([mask, torch.zeros(self.max_objects - len(mask))])
        elif len(mask) > self.max_objects:
            mask = mask[:self.max_objects]
        
        # Get question and answer
        question = sample.get('question', '')
        answer = sample.get('answer', '')
        
        # Get numerical targets if available
        numerical_targets = sample.get('numerical_targets', None)
        if numerical_targets is None:
            numerical_targets = torch.zeros(6)  # Default 6 numerical outputs
        elif isinstance(numerical_targets, dict):
            # Convert dict to tensor
            targets = torch.zeros(6)
            if 'distance' in numerical_targets:
                targets[0] = numerical_targets['distance']
            if 'speed' in numerical_targets:
                targets[1] = numerical_targets['speed']
            if 'time_to_collision' in numerical_targets:
                targets[2] = numerical_targets['time_to_collision']
            if 'kinetic_energy' in numerical_targets:
                targets[3] = numerical_targets['kinetic_energy']
            if 'momentum' in numerical_targets:
                targets[4] = numerical_targets['momentum']
            if 'object_count' in numerical_targets:
                targets[5] = numerical_targets['object_count']
            numerical_targets = targets
        elif not isinstance(numerical_targets, torch.Tensor):
            numerical_targets = torch.tensor(numerical_targets, dtype=torch.float32)
        
        return {
            'physics_states': states,
            'object_mask': mask,
            'questions': question,
            'answers': answer,
            'numerical_targets': numerical_targets
        }

# ============================================================
# CREATE TRAIN/TEST SPLIT AND DATALOADERS
# ============================================================
print("\nCreating train/test split...")

full_dataset = AdapterQADataset(qa_dataset_raw, max_objects=MAX_OBJECTS, state_dim=STATE_DIM)

# 90/10 train/test split
train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"  Train samples: {len(train_dataset):,}")
print(f"  Test samples: {len(test_dataset):,}")

# Custom collate function to handle string fields
def collate_fn(batch):
    physics_states = torch.stack([b['physics_states'] for b in batch])
    object_mask = torch.stack([b['object_mask'] for b in batch])
    questions = [b['questions'] for b in batch]
    answers = [b['answers'] for b in batch]
    numerical_targets = torch.stack([b['numerical_targets'] for b in batch])
    
    return {
        'physics_states': physics_states,
        'object_mask': object_mask,
        'questions': questions,
        'answers': answers,
        'numerical_targets': numerical_targets
    }

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader):,}")
print(f"  Test batches: {len(test_loader):,}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Workers: {NUM_WORKERS}")
print("✓ Ready for training!")

In [ ]:
# ============================================================
# CELL 13: TRAINING FUNCTIONS WITH PHYSICS V2 TECHNOLOGIES
# ============================================================
# Features from train_physics_former_v2.ipynb:
# - AMP (Automatic Mixed Precision) for faster training
# - OneCycleLR scheduler for better convergence
# - PlateauTracker + Catapult for plateau breakthrough
# - Diversity penalty / contrastive loss (model's compute_combined_loss_with_contrastive)
# - Physics usage validation
# - Catapult resets early stopping patience (plateau epochs don't count against progression)
# - SAMPLE DISPLAY: Shows actual questions and model answers for validation

from torch.optim.lr_scheduler import OneCycleLR

# ============================================================
# AMP CONFIGURATION
# ============================================================
USE_AMP = torch.cuda.is_available()  # Only use AMP on CUDA

if USE_AMP:
    scaler = torch.cuda.amp.GradScaler()
    print("AMP: ENABLED (GradScaler initialized)")
else:
    scaler = None
    print("AMP: DISABLED (CPU mode)")

# Physics validation configuration
VALIDATE_PHYSICS_EVERY = 3  # Validate physics usage every N epochs
PHYSICS_SIM_WARNING_THRESHOLD = 0.95  # Warn if cosine similarity > this (model ignoring physics)


# ============================================================
# PLATEAU BREAKTHROUGH (from physics former v2)
# ============================================================
class PlateauTracker:
    """
    Detects training plateaus and triggers learning rate catapults.
    Based on 2025 research showing transformers learn in "bursts" after plateaus.

    Key behavior: When catapult triggers, it signals to reset early stopping patience.
    This gives the catapult a fair chance to work before stopping training.
    """
    def __init__(self, patience: int = 5, min_delta: float = 0.003):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.plateau_count = 0
        self.catapult_countdown = 0
        self.catapult_lr = None
        self.base_lr = None
        self.catapult_just_triggered = False  # Flag for resetting early stopping

    def update(self, loss: float) -> bool:
        """Returns True if plateau detected."""
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.plateau_count = 0
            return False
        else:
            self.plateau_count += 1
            return self.plateau_count >= self.patience

    def trigger_catapult(self, optimizer, multiplier: float = 1.5, duration: int = 3):
        """Trigger LR catapult to escape plateau."""
        self.base_lr = optimizer.param_groups[0]['lr']
        config_base_lr = PHASE1_LR  # Use phase 1 LR as base
        self.catapult_lr = config_base_lr * multiplier
        self.catapult_countdown = duration
        self.catapult_just_triggered = True  # Signal to reset early stopping
        self.plateau_count = 0  # Reset plateau counter

        for param_group in optimizer.param_groups:
            param_group['lr'] = self.catapult_lr
        print(f"    🚀 CATAPULT: LR boosted to {self.catapult_lr:.6f} for {duration} epochs")
        print(f"    🚀 CATAPULT: Early stopping patience RESET (giving catapult a fair chance)")

    def enforce_catapult_lr(self, optimizer):
        """Call AFTER scheduler.step() to override with catapult LR."""
        if self.catapult_countdown > 0 and self.catapult_lr is not None:
            for param_group in optimizer.param_groups:
                param_group['lr'] = self.catapult_lr
            self.catapult_countdown -= 1
            if self.catapult_countdown == 0:
                print(f"    🚀 CATAPULT: Ended, returning to scheduler LR")
                self.catapult_lr = None

    def should_reset_early_stopping(self) -> bool:
        """Check and consume the catapult trigger flag."""
        if self.catapult_just_triggered:
            self.catapult_just_triggered = False
            return True
        return False

    def is_in_catapult(self) -> bool:
        """Check if currently in catapult period (don't count against early stopping)."""
        return self.catapult_countdown > 0


# ============================================================
# PHYSICS USAGE VALIDATION
# ============================================================
def validate_physics_usage(model, val_loader, device, num_samples=20):
    """
    Validate that the model is actually using physics information.

    Compares prefix tokens between real physics and zero physics inputs.
    Returns cosine similarity (lower = better physics usage).

    If similarity > 0.95, the model is likely ignoring physics input!
    """
    model.eval()
    similarities = []
    differences = []

    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_samples:
                break

            states = batch['physics_states'].to(device)
            masks = batch['object_mask'].to(device)
            zero_states = torch.zeros_like(states)

            # Get prefix tokens for real and zero physics
            real_features = model.extract_physics_features(states, masks)
            zero_features = model.extract_physics_features(zero_states, masks)

            real_prefix = model.create_prefix_tokens(real_features)
            zero_prefix = model.create_prefix_tokens(zero_features)

            # Flatten and compute cosine similarity
            real_flat = real_prefix.view(real_prefix.size(0), -1)
            zero_flat = zero_prefix.view(zero_prefix.size(0), -1)

            cos_sim = torch.nn.functional.cosine_similarity(real_flat, zero_flat, dim=1)
            diff_norm = (real_flat - zero_flat).norm(dim=1)

            similarities.extend(cos_sim.cpu().tolist())
            differences.extend(diff_norm.cpu().tolist())

    avg_similarity = sum(similarities) / len(similarities) if similarities else 1.0
    avg_difference = sum(differences) / len(differences) if differences else 0.0

    return {
        'avg_cosine_similarity': avg_similarity,
        'avg_difference_norm': avg_difference,
        'num_samples': len(similarities)
    }


# ============================================================
# SAMPLE DISPLAY - Show actual Q&A for validation
# ============================================================
def display_validation_samples(model, val_loader, device, num_samples=5):
    """
    Display actual questions and model answers to verify the model is working.
    Shows question, expected answer, model prediction, and correctness.
    """
    model.eval()
    print("\n" + "=" * 70)
    print("VALIDATION SAMPLES - Actual Questions and Answers")
    print("=" * 70)

    samples_shown = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            if samples_shown >= num_samples:
                break

            physics_states = batch['physics_states'].to(device)
            object_mask = batch['object_mask'].to(device)
            questions = batch['questions']
            answers = batch['answers']

            # Get model predictions
            predictions = model(physics_states, object_mask, questions)

            # Also get descriptive head predictions for comparison
            features = model.extract_physics_features(physics_states, object_mask)
            desc_preds = model.descriptive_head.predict_batch(features, questions)

            for i, (q, expected, pred, desc_pred) in enumerate(zip(questions, answers, predictions, desc_preds)):
                if samples_shown >= num_samples:
                    break

                # Clean predictions
                pred_clean = pred.split('\n')[0].strip().lower()
                exp_clean = expected.strip().lower()

                # Check correctness
                is_correct = (
                    pred_clean == exp_clean or
                    exp_clean in pred_clean or
                    pred_clean in exp_clean
                )

                status = "✓ CORRECT" if is_correct else "✗ WRONG"
                if is_correct:
                    correct += 1
                total += 1

                print(f"\n[Sample {samples_shown + 1}] {status}")
                print(f"  Q: {q[:80]}{'...' if len(q) > 80 else ''}")
                print(f"  Expected: {expected}")
                print(f"  LLM Pred: {pred[:60]}{'...' if len(pred) > 60 else ''}")
                print(f"  Desc Head: {desc_pred}")

                samples_shown += 1

    accuracy = 100.0 * correct / max(total, 1)
    print(f"\n{'=' * 70}")
    print(f"Sample Accuracy: {correct}/{total} ({accuracy:.1f}%)")
    print("=" * 70)
    return accuracy


# ============================================================
# TRAINING EPOCH WITH AMP + MODEL'S COMPUTE_COMBINED_LOSS_WITH_CONTRASTIVE
# ============================================================
def train_epoch_v2(adapter, dataloader, optimizer, scheduler, device,
                   use_contrastive=True, contrastive_weight=0.1,
                   use_amp=True, scaler=None, plateau_tracker=None):
    """
    Train for one epoch using model's built-in compute_combined_loss_with_contrastive.

    Features:
    - AMP (Automatic Mixed Precision) for faster training on GPU
    - OneCycleLR scheduler support
    - PlateauTracker integration for catapult
    - Uses adapter.compute_combined_loss_with_contrastive()
    """
    adapter.train()
    total_loss = 0.0
    cat_loss_sum = 0.0
    num_loss_sum = 0.0
    desc_loss_sum = 0.0
    contr_loss_sum = 0.0
    num_batches = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch in pbar:
        physics_states = batch['physics_states'].to(device)
        object_mask = batch['object_mask'].to(device)
        questions = batch['questions']
        answers = batch['answers']
        choices = batch.get('choices')
        correct_choice_idx = batch.get('correct_choice_idx')
        if correct_choice_idx is not None:
            correct_choice_idx = correct_choice_idx.to(device)
        numerical_targets = batch['numerical_targets'].to(device)

        optimizer.zero_grad()

        # Use AMP autocast if enabled
        if use_amp and scaler is not None:
            with torch.cuda.amp.autocast():
                if use_contrastive:
                    loss, loss_dict = adapter.compute_combined_loss_with_contrastive(
                        physics_states, object_mask, questions, answers,
                        choices=choices,
                        correct_choice_idx=correct_choice_idx,
                        numerical_targets=numerical_targets,
                        categorical_weight=1.0,
                        numerical_weight=0.5,
                        contrastive_weight=contrastive_weight
                    )
                else:
                    loss, loss_dict = adapter.compute_combined_loss(
                        physics_states, object_mask, questions, answers,
                        choices=choices,
                        correct_choice_idx=correct_choice_idx,
                        numerical_targets=numerical_targets,
                        categorical_weight=1.0,
                        numerical_weight=0.5
                    )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            # Non-AMP path
            if use_contrastive:
                loss, loss_dict = adapter.compute_combined_loss_with_contrastive(
                    physics_states, object_mask, questions, answers,
                    choices=choices,
                    correct_choice_idx=correct_choice_idx,
                    numerical_targets=numerical_targets,
                    categorical_weight=1.0,
                    numerical_weight=0.5,
                    contrastive_weight=contrastive_weight
                )
            else:
                loss, loss_dict = adapter.compute_combined_loss(
                    physics_states, object_mask, questions, answers,
                    choices=choices,
                    correct_choice_idx=correct_choice_idx,
                    numerical_targets=numerical_targets,
                    categorical_weight=1.0,
                    numerical_weight=0.5
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            optimizer.step()

        # Step scheduler (OneCycleLR steps per batch)
        if scheduler is not None:
            scheduler.step()
            # Enforce catapult LR if active
            if plateau_tracker is not None:
                plateau_tracker.enforce_catapult_lr(optimizer)

        # Accumulate losses
        total_loss += loss.item()
        cat_loss_sum += loss_dict['categorical'].item()
        num_loss_sum += loss_dict['numerical'].item()
        desc_loss = loss_dict.get('descriptive', torch.tensor(0.0)).item()
        desc_loss_sum += desc_loss
        contr_loss = loss_dict.get('contrastive', torch.tensor(0.0)).item()
        contr_loss_sum += contr_loss
        num_batches += 1

        # Update progress bar with detailed loss breakdown
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'cat': f'{loss_dict["categorical"].item():.3f}',
            'num': f'{loss_dict["numerical"].item():.3f}',
            'contr': f'{contr_loss:.3f}'
        })

    return {
        'total_loss': total_loss / max(num_batches, 1),
        'categorical_loss': cat_loss_sum / max(num_batches, 1),
        'numerical_loss': num_loss_sum / max(num_batches, 1),
        'descriptive_loss': desc_loss_sum / max(num_batches, 1),
        'contrastive_loss': contr_loss_sum / max(num_batches, 1),
        'num_batches': num_batches
    }


# ============================================================
# TRAINING PHASE WITH ALL V2 TECHNOLOGIES
# ============================================================
def train_phase_v2(adapter, train_loader, test_loader, optimizer, device,
                   phase_name, max_epochs=100, patience=5, min_delta=0.001,
                   checkpoint_dir=None, phase_num=1, save_every_n_epochs=3,
                   use_contrastive=True, contrastive_weight=0.1,
                   validate_every_n_epochs=5, physics_sim_threshold=0.95,
                   use_amp=True, scaler=None, use_plateau_tracker=True,
                   show_samples_every_n_epochs=3):
    """
    Train with all Physics Former V2 technologies:
    - AMP (Automatic Mixed Precision)
    - OneCycleLR scheduler (per-epoch)
    - PlateauTracker + Catapult (with patience reset)
    - Physics usage validation
    - Model's compute_combined_loss_with_contrastive
    - SAMPLE DISPLAY: Shows actual Q&A every N epochs

    Key: Plateau epochs DON'T count against level progression.
    When catapult triggers, early stopping patience is reset.
    """
    print(f"\n{'=' * 70}")
    print(f"TRAINING PHASE: {phase_name}")
    print(f"{'=' * 70}")
    print(f"  Early stopping: patience={patience}, min_delta={min_delta}")
    print(f"  Contrastive loss: {'ENABLED' if use_contrastive else 'DISABLED'} (weight={contrastive_weight})")
    print(f"  AMP: {'ENABLED' if use_amp and scaler else 'DISABLED'}")
    print(f"  OneCycleLR: Per-epoch scheduler")
    print(f"  PlateauTracker: {'ENABLED' if use_plateau_tracker else 'DISABLED'} (resets early stopping on catapult)")
    print(f"  Physics validation: every {validate_every_n_epochs} epochs (warn if sim > {physics_sim_threshold})")
    print(f"  Sample display: every {show_samples_every_n_epochs} epochs")
    if checkpoint_dir:
        print(f"  Checkpoints: {checkpoint_dir}")
    print("=" * 70)

    best_loss = float('inf')
    epochs_without_improvement = 0
    plateau_tracker = PlateauTracker(patience=3, min_delta=0.003) if use_plateau_tracker else None

    for epoch in range(max_epochs):
        # Create OneCycleLR scheduler for this epoch
        scheduler = OneCycleLR(
            optimizer,
            max_lr=optimizer.param_groups[0]['lr'] * 2,  # Peak at 2x current LR
            steps_per_epoch=len(train_loader),
            epochs=1
        )

        # Training
        loss_dict = train_epoch_v2(
            adapter, train_loader, optimizer, scheduler, device,
            use_contrastive=use_contrastive,
            contrastive_weight=contrastive_weight,
            use_amp=use_amp,
            scaler=scaler,
            plateau_tracker=plateau_tracker
        )

        avg_loss = loss_dict['total_loss']

        # Check for plateau and trigger catapult
        catapult_triggered = False
        if plateau_tracker is not None:
            if plateau_tracker.update(avg_loss):
                plateau_tracker.trigger_catapult(optimizer, multiplier=1.5, duration=3)
                catapult_triggered = True

            # Reset early stopping patience when catapult triggers
            if plateau_tracker.should_reset_early_stopping():
                epochs_without_improvement = 0
                print(f"    → Early stopping patience reset to 0/{patience}")

        # Check for improvement (only count if NOT in catapult period)
        improved = False
        in_catapult = plateau_tracker.is_in_catapult() if plateau_tracker else False

        if best_loss - avg_loss > min_delta:
            best_loss = avg_loss
            epochs_without_improvement = 0
            status = "improved ✓"
            improved = True
        elif in_catapult:
            # During catapult, don't increment early stopping counter
            status = f"catapult active 🚀 (not counting against patience)"
        else:
            epochs_without_improvement += 1
            status = f"no improvement ({epochs_without_improvement}/{patience})"

        # Log epoch results with detailed loss breakdown
        lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch+1:3d}/{max_epochs}")
        print(f"  Loss: {avg_loss:.4f} (best: {best_loss:.4f}) | {status}")
        print(f"  Breakdown: cat={loss_dict['categorical_loss']:.4f} | num={loss_dict['numerical_loss']:.4f} | "
              f"desc={loss_dict['descriptive_loss']:.4f} | contr={loss_dict['contrastive_loss']:.4f}")
        print(f"  LR: {lr:.2e}")

        # Show sample Q&A every N epochs
        if (epoch + 1) % show_samples_every_n_epochs == 0:
            display_validation_samples(adapter, test_loader, device, num_samples=5)

        # Physics usage validation every N epochs
        if (epoch + 1) % validate_every_n_epochs == 0:
            print("\n  [PHYSICS VALIDATION]")
            val_results = validate_physics_usage(adapter, test_loader, device)
            sim = val_results['avg_cosine_similarity']
            diff = val_results['avg_difference_norm']
            print(f"    Prefix cosine similarity (real vs zero): {sim:.4f}")
            print(f"    Prefix difference norm: {diff:.2f}")
            if sim > physics_sim_threshold:
                print(f"    ⚠️  WARNING: High similarity suggests model NOT using physics!")
            elif sim < 0.8:
                print(f"    ✓ GOOD: Low similarity suggests model IS using physics!")

        # Save checkpoint
        if checkpoint_dir and ((epoch + 1) % save_every_n_epochs == 0 or improved):
            ckpt_path = Path(checkpoint_dir) / f"adapter_phase{phase_num}_epoch{epoch+1}_loss{avg_loss:.4f}.pt"
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                'model_state_dict': adapter.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'phase': phase_num,
                'epoch': epoch + 1,
                'loss': avg_loss,
                'best_loss': best_loss,
            }, ckpt_path)
            print(f"  → Checkpoint saved: {ckpt_path.name}")

        # Only trigger early stopping if not in catapult period
        if epochs_without_improvement >= patience and not in_catapult:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

    # Final physics validation and sample display
    print("\n" + "=" * 70)
    print("PHASE COMPLETE - FINAL VALIDATION")
    print("=" * 70)

    val_results = validate_physics_usage(adapter, test_loader, device)
    print(f"  Physics prefix cosine similarity: {val_results['avg_cosine_similarity']:.4f}")
    print(f"  Physics prefix difference norm: {val_results['avg_difference_norm']:.2f}")

    display_validation_samples(adapter, test_loader, device, num_samples=10)

    # Save phase-end checkpoint
    if checkpoint_dir:
        phase_end_path = Path(checkpoint_dir) / f"adapter_phase{phase_num}_complete_loss{best_loss:.4f}.pt"
        torch.save({
            'model_state_dict': adapter.state_dict(),
            'phase': phase_num,
            'loss': best_loss,
        }, phase_end_path)
        print(f"  → Phase complete checkpoint: {phase_end_path.name}")

    print(f"\n{phase_name} complete! Best loss: {best_loss:.4f}")
    return best_loss


# ============================================================
# EVALUATION WITH SAMPLE DISPLAY
# ============================================================
def evaluate(adapter, dataloader, device, num_samples=100, show_samples=True, num_display=10):
    """Evaluate adapter accuracy with optional sample display."""
    adapter.eval()
    correct = 0
    total = 0
    samples_to_show = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            if total >= num_samples:
                break

            physics_states = batch['physics_states'].to(device)
            object_mask = batch['object_mask'].to(device)
            questions = batch['questions']
            answers = batch['answers']

            predictions = adapter(physics_states, object_mask, questions)

            for pred, expected, q in zip(predictions, answers, questions):
                pred_clean = pred.split('\n')[0].strip().lower()
                exp_clean = expected.strip().lower()

                is_correct = (
                    pred_clean == exp_clean or
                    exp_clean in pred_clean or
                    pred_clean in exp_clean
                )

                if is_correct:
                    correct += 1

                if len(samples_to_show) < num_display:
                    samples_to_show.append({
                        'question': q,
                        'expected': expected,
                        'predicted': pred,
                        'correct': is_correct
                    })

                total += 1

    accuracy = 100.0 * correct / max(total, 1)

    if show_samples and samples_to_show:
        print("\n" + "=" * 70)
        print("EVALUATION SAMPLES")
        print("=" * 70)
        for i, s in enumerate(samples_to_show):
            status = "✓" if s['correct'] else "✗"
            print(f"\n[{i+1}] {status}")
            print(f"  Q: {s['question'][:70]}{'...' if len(s['question']) > 70 else ''}")
            print(f"  Expected: {s['expected']}")
            print(f"  Predicted: {s['predicted'][:50]}{'...' if len(s['predicted']) > 50 else ''}")
        print("=" * 70)

    return accuracy


print("\n" + "=" * 70)
print("TRAINING TECHNOLOGIES LOADED (Physics Former V2)")
print("=" * 70)
print("  ✓ AMP (Automatic Mixed Precision) - 2x faster on GPU")
print("  ✓ OneCycleLR scheduler - Better convergence")
print("  ✓ PlateauTracker + Catapult - Escape training plateaus")
print("  ✓ Catapult resets early stopping - Plateau epochs don't count against progression")
print("  ✓ compute_combined_loss_with_contrastive() - Model's built-in loss")
print("  ✓ Physics usage validation - Detect modality collapse")
print("  ✓ Sample display - See actual Q&A during training")
print(f"  ✓ Physics similarity threshold: {PHYSICS_SIM_WARNING_THRESHOLD}")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 14: PHASE 1 - TRAIN ADAPTER + HEADS
# ============================================================
print("="*70)
print("PHASE 1: Training Adapter + Heads (LLM frozen)")
print("="*70)

# Ensure LLM is frozen
adapter.set_training_phase('adapter')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE1_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE1_LR}")

# Use train_phase_v2 with all technologies
train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 1: Adapter + Numerical Head",
    max_epochs=20,
    patience=5,
    min_delta=0.001,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=1,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True
)

In [ ]:
# ============================================================
# CELL 15: PHASE 2 - UNFREEZE LLM OUTPUT LAYER
# ============================================================
print("="*70)
print("PHASE 2: Training + LLM Output Layer")
print("="*70)

# Unfreeze LLM output layer
adapter.set_training_phase('llm_head')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE2_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE2_LR}")

train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 2: LLM Head",
    max_epochs=15,
    patience=4,
    min_delta=0.0005,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=2,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True
)

In [ ]:
# ============================================================
# CELL 16: PHASE 3 - FULL LLM FINE-TUNING
# ============================================================
print("="*70)
print("PHASE 3: Full LLM Fine-tuning (DistilGPT-2)")
print("="*70)

# Unfreeze full LLM
adapter.set_training_phase('full')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE3_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE3_LR}")

train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 3: Full LLM (DistilGPT-2)",
    max_epochs=10,
    patience=3,
    min_delta=0.0005,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=3,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True
)

In [ ]:
# ============================================================
# CELL 17: FINAL EVALUATION
# ============================================================
print("="*70)
print("FINAL EVALUATION")
print("="*70)

accuracy = evaluate(adapter, test_loader, device, num_samples=500)
print(f"\nTest accuracy: {accuracy:.1f}%")

# Save final model
final_path = Path(OUTPUT_DIR) / "adapter_v2_final.pt"
torch.save({
    'model_state_dict': adapter.state_dict(),
    'physics_dim': hidden_dim,
    'num_prefix_tokens': NUM_PREFIX_TOKENS,
    'llm_name': 'distilgpt2',
    'accuracy': accuracy,
    'question_types': [qt.value for qt in PHYSICS_QUESTION_TYPES]
}, final_path)

print(f"\nFinal model saved: {final_path}")
print(f"Total parameters: {sum(p.numel() for p in adapter.parameters()):,}")

In [ ]:
# ============================================================
# CELL 18: TEST INFERENCE
# ============================================================
print("="*70)
print("TEST INFERENCE")
print("="*70)

# Get a sample batch
adapter.eval()
sample_batch = next(iter(test_loader))

physics_states = sample_batch['physics_states'].to(device)
object_mask = sample_batch['object_mask'].to(device)
questions = sample_batch['questions']
answers = sample_batch['answers']

with torch.no_grad():
    predictions = adapter(physics_states, object_mask, questions)

print("\nSample predictions:")
print("-" * 70)
for i in range(min(5, len(questions))):
    q = questions[i]
    expected = answers[i]
    predicted = predictions[i]
    match = "✓" if predicted.lower().strip() == expected.lower().strip() else "✗"
    print(f"Q: {q[:60]}...")
    print(f"   Expected: {expected}")
    print(f"   Predicted: {predicted} {match}")
    print()